<a href="https://colab.research.google.com/github/gulzhanmsc/IB9AU/blob/main/Gen_AI_Task_5.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

Gulzhan Aitileu

Key insights:

This code builds a semantic search engine for financial news instead of matching keywords, it understands the meaning of a query and finds relevant articles even if they use different words.
The raw text had URLs mixed into the headlines, so I used regex to extract and remove them. r'https?://\S+' describes the pattern, re.findall() collects matches, and re.sub() strips them out. Returning a pd.Series from apply() was a neat trick to split one column into two in a single step.
Embeddings represent text meaning as vectors, similar sentences get similar vectors. I used the pre-trained all-MiniLM-L6-v2, so no training was needed. Batching articles in groups of 64 kept encoding efficient. For search, the query gets embedded the same way, then compared to every article vector via cosine similarity. np.argsort()[::-1] was a clean pattern for ranking results.
The Gradio UI took surprisingly little code gr.Blocks() handled layout, query.submit() made pressing Enter trigger the search, and share=True generated a public link instantly without any deployment.

In [1]:
import pandas as pd
import numpy as np
import re #Regular Expressions - Python library for finding and manipulating patterns in text
from sklearn.metrics.pairwise import cosine_similarity #sklearn (scikit-learn) — a large machine learning library. cosine_similarity — to measure how close two vectors are to each other in meaning.
from sentence_transformers import SentenceTransformer #a library built specifically for turning text into embeddings using pre-trained transformer models. This is the heart of project — it's what makes the search semantic rather than just keyword-based.
import gradio as gr #a library for building quick interactive web interfaces for Python functions.

In [2]:
# Accessing the data from Google Drive
from google.colab import drive
drive.mount('/content/drive')

Mounted at /content/drive


In [3]:
# Accessing the data from Google Drive
df = pd.read_csv('/content/drive/MyDrive/Gen AI Data/financial_news.csv')

In [4]:
# Extract URLs and clean text
def extract_urls(text):
    urls = re.findall(r'https?://\S+', str(text))
    clean = re.sub(r'https?://\S+', '', str(text)).strip()
    return pd.Series([re.sub(r'  +', ' ', clean), urls[0] if urls else None])

df[['text', 'URL']] = df['text'].apply(extract_urls)


In [5]:
# Build embeddings
model = SentenceTransformer('all-MiniLM-L6-v2')
embeddings = model.encode(df['text'].tolist(), batch_size=64, show_progress_bar=True)

/usr/local/lib/python3.12/dist-packages/huggingface_hub/utils/_auth.py:94: UserWarning: 
The secret `HF_TOKEN` does not exist in your Colab secrets.
To authenticate with the Hugging Face Hub, create a token in your settings tab (https://huggingface.co/settings/tokens), set it as secret in your Google Colab and restart your session.
You will be able to reuse this secret in all of your notebooks.
Please note that authentication is recommended but still optional to access public models or datasets.
  warnings.warn(


modules.json:   0%|          | 0.00/349 [00:00<?, ?B/s]

config_sentence_transformers.json:   0%|          | 0.00/116 [00:00<?, ?B/s]

README.md: 0.00B [00:00, ?B/s]

sentence_bert_config.json:   0%|          | 0.00/53.0 [00:00<?, ?B/s]

...

vocab.txt: 0.00B [00:00, ?B/s]

tokenizer.json: 0.00B [00:00, ?B/s]

special_tokens_map.json:   0%|          | 0.00/112 [00:00<?, ?B/s]

config.json:   0%|          | 0.00/190 [00:00<?, ?B/s]

Batches:   0%|          | 0/266 [00:00<?, ?it/s]

In [6]:
# Search function
def search(query, top_k=5):
    scores = cosine_similarity(model.encode([query]), embeddings)[0]
    idx = np.argsort(scores)[::-1][:top_k]
    results = df.iloc[idx][['text', 'URL']].copy()
    results.insert(0, 'Score', scores[idx].round(4))
    return results.reset_index(drop=True)

In [7]:
# Gradio UI
with gr.Blocks(title="Financial News Search") as demo:
    gr.Markdown("## 📰 Financial News Semantic Search")
    query = gr.Textbox(placeholder="e.g. earnings surprise, regulatory fine …")
    btn = gr.Button("Search", variant="primary")
    out = gr.Dataframe(headers=["Score", "Headline", "URL"], wrap=True)
    btn.click(fn=search, inputs=query, outputs=out)
    query.submit(fn=search, inputs=query, outputs=out)

demo.launch(share=True)

Colab notebook detected. To show errors in colab notebook, set debug=True in launch()
* Running on public URL: https://7308f72485a97f5738.gradio.live

This share link expires in 1 week. For free permanent hosting and GPU upgrades, run `gradio deploy` from the terminal in the working directory to deploy to Hugging Face Spaces (https://huggingface.co/spaces)
